# GCT-VAL-001 Validation Notebook (DuckDB)

R001-R015 validations with BLOCKER precedence and manifest-driven rule coverage.

In [1]:
from pathlib import Path
from datetime import datetime
import json
import re
import duckdb

ROOT = Path.cwd()
if not (ROOT / 'data_raw').exists():
    ROOT = ROOT.parent

DATA_RAW = ROOT / 'data_raw'
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(parents=True, exist_ok=True)

RUN_DATE = '2026-05-10'
datetime.fromisoformat(RUN_DATE)

MANIFEST_PATH = OUTPUTS / 'injection_manifest.json'

print(f'ROOT={ROOT}')
print(f'RUN_DATE={RUN_DATE}')
print(f'MANIFEST_PATH={MANIFEST_PATH}')

ROOT=C:\Users\halol\Desktop\encounters
RUN_DATE=2026-05-10
MANIFEST_PATH=C:\Users\halol\Desktop\encounters\outputs\injection_manifest.json


In [2]:
con = duckdb.connect()

header_csv = (DATA_RAW / 'encounters_header.csv').as_posix()
line_csv = (DATA_RAW / 'encounters_lines.csv').as_posix()
member_csv = (DATA_RAW / 'reference_members.csv').as_posix()
provider_csv = (DATA_RAW / 'reference_providers.csv').as_posix()
rejects_csv = (OUTPUTS / 'rejects.csv').as_posix()
triage_csv = (OUTPUTS / 'triage_summary.csv').as_posix()
triage_by_code_csv = (OUTPUTS / 'triage_by_code.csv').as_posix()

con.execute(f"""
CREATE OR REPLACE TEMP VIEW header_raw AS
SELECT * FROM read_csv_auto('{header_csv}', HEADER=TRUE, ALL_VARCHAR=TRUE);

CREATE OR REPLACE TEMP VIEW line_raw AS
SELECT * FROM read_csv_auto('{line_csv}', HEADER=TRUE, ALL_VARCHAR=TRUE);

CREATE OR REPLACE TEMP VIEW member_raw AS
SELECT * FROM read_csv_auto('{member_csv}', HEADER=TRUE, ALL_VARCHAR=TRUE);

CREATE OR REPLACE TEMP VIEW provider_raw AS
SELECT * FROM read_csv_auto('{provider_csv}', HEADER=TRUE, ALL_VARCHAR=TRUE);

-- Canonical missingness predicate required by contract:
--   IS NULL OR TRIM(col) = ''
CREATE OR REPLACE TEMP MACRO is_missing(x) AS (x IS NULL OR trim(x) = '');

CREATE OR REPLACE TEMP VIEW header_norm AS
SELECT
  trim(batch_id) AS batch_id,
  trim(claim_id) AS claim_id,
  trim(member_id) AS member_id,
  trim(provider_npi) AS provider_npi,
  trim(lob) AS lob,
  trim(vendor) AS vendor,
  trim(service_from) AS service_from,
  trim(service_to) AS service_to,
  trim(total_charge) AS total_charge,
  trim(total_allowed) AS total_allowed,
  trim(total_paid) AS total_paid,
  trim(adjudication_date) AS adjudication_date,
  TRY_CAST(NULLIF(trim(service_from), '') AS DATE) AS service_from_dt,
  TRY_CAST(NULLIF(trim(service_to), '') AS DATE) AS service_to_dt,
  CAST(ROUND(TRY_CAST(NULLIF(trim(total_charge), '') AS DOUBLE), 2) AS DECIMAL(18,2)) AS total_charge_2dp,
  CAST(ROUND(TRY_CAST(NULLIF(trim(total_allowed), '') AS DOUBLE), 2) AS DECIMAL(18,2)) AS total_allowed_2dp,
  CAST(ROUND(TRY_CAST(NULLIF(trim(total_paid), '') AS DOUBLE), 2) AS DECIMAL(18,2)) AS total_paid_2dp
FROM header_raw;

CREATE OR REPLACE TEMP VIEW line_norm AS
SELECT
  trim(claim_id) AS claim_id,
  trim(line_id) AS line_id,
  trim(proc_code) AS proc_code,
  trim(dx_code) AS dx_code,
  trim(units) AS units,
  trim(charge) AS charge,
  trim(allowed) AS allowed,
  trim(paid) AS paid,
  CAST(ROUND(TRY_CAST(NULLIF(trim(charge), '') AS DOUBLE), 2) AS DECIMAL(18,2)) AS charge_2dp,
  CAST(ROUND(TRY_CAST(NULLIF(trim(allowed), '') AS DOUBLE), 2) AS DECIMAL(18,2)) AS allowed_2dp,
  CAST(ROUND(TRY_CAST(NULLIF(trim(paid), '') AS DOUBLE), 2) AS DECIMAL(18,2)) AS paid_2dp
FROM line_raw;

CREATE OR REPLACE TEMP VIEW member_norm AS
SELECT
  trim(member_id) AS member_id,
  TRY_CAST(NULLIF(trim(coverage_start), '') AS DATE) AS coverage_start_dt,
  TRY_CAST(NULLIF(trim(coverage_end), '') AS DATE) AS coverage_end_dt
FROM member_raw;

CREATE OR REPLACE TEMP VIEW provider_norm AS
SELECT trim(provider_npi) AS provider_npi
FROM provider_raw;

-- BLOCKER precedence anchor:
-- assign at most one BLOCKER rule per header row by explicit ordered CASE.
CREATE OR REPLACE TEMP VIEW header_with_blocker AS
SELECT
  h.*,
  CASE
    WHEN is_missing(h.member_id) THEN 'R001'
    WHEN is_missing(h.claim_id) THEN 'R002'
    WHEN is_missing(h.provider_npi) THEN 'R003'
    WHEN is_missing(h.service_from) THEN 'R004'
    WHEN NOT is_missing(h.provider_npi)
      AND regexp_full_match(h.provider_npi, '^[0-9]+$')
      AND length(h.provider_npi) <> 10 THEN 'R005'
    WHEN NOT is_missing(h.provider_npi)
      AND NOT regexp_full_match(h.provider_npi, '^[0-9]+$') THEN 'R006'
    WHEN NOT is_missing(h.provider_npi)
      AND regexp_full_match(h.provider_npi, '^[0-9]+$')
      AND length(h.provider_npi) = 10
      AND NOT EXISTS (SELECT 1 FROM provider_norm p WHERE p.provider_npi = h.provider_npi) THEN 'R007'
    WHEN h.service_from_dt IS NOT NULL
      AND h.service_to_dt IS NOT NULL
      AND h.service_to_dt < h.service_from_dt THEN 'R008'
    WHEN (h.service_from_dt IS NOT NULL AND h.service_from_dt > DATE '{RUN_DATE}')
      OR (h.service_to_dt IS NOT NULL AND h.service_to_dt > DATE '{RUN_DATE}') THEN 'R009'
    ELSE NULL
  END AS blocker_rule
FROM header_norm h;

CREATE OR REPLACE TEMP VIEW clean_header AS
SELECT *
FROM header_with_blocker
WHERE blocker_rule IS NULL;

CREATE OR REPLACE TEMP VIEW rejects_header_blocker AS
SELECT
  h.claim_id,
  CAST(NULL AS VARCHAR) AS line_id,
  h.batch_id,
  h.lob,
  h.vendor,
  h.service_from,
  CASE
    WHEN h.blocker_rule IN ('R001','R002','R003','R004') THEN 'REQUIRED_FIELD'
    WHEN h.blocker_rule IN ('R005','R006','R007') THEN 'PROVIDER'
    WHEN h.blocker_rule IN ('R008','R009') THEN 'DATES'
  END AS reject_category,
  CASE h.blocker_rule
    WHEN 'R001' THEN 'NULL_MEMBER_ID'
    WHEN 'R002' THEN 'NULL_CLAIM_ID'
    WHEN 'R003' THEN 'NULL_PROVIDER_NPI'
    WHEN 'R004' THEN 'NULL_SERVICE_FROM'
    WHEN 'R005' THEN 'NPI_BAD_LENGTH'
    WHEN 'R006' THEN 'NPI_NOT_NUMERIC'
    WHEN 'R007' THEN 'NPI_NOT_FOUND'
    WHEN 'R008' THEN 'SERVICE_TO_BEFORE_FROM'
    WHEN 'R009' THEN 'FUTURE_SERVICE_DATE'
  END AS reject_code,
  'BLOCKER' AS severity,
  strftime(current_timestamp, '%Y-%m-%dT%H:%M:%SZ') AS detected_ts,
  h.blocker_rule AS rule_id
FROM header_with_blocker h
WHERE h.blocker_rule IS NOT NULL;

CREATE OR REPLACE TEMP VIEW rejects_header_non_blocker AS
WITH r010 AS (
  SELECT
    h.claim_id,
    CAST(NULL AS VARCHAR) AS line_id,
    h.batch_id,
    h.lob,
    h.vendor,
    h.service_from,
    'MEMBER_ELIGIBILITY' AS reject_category,
    'SERVICE_OUTSIDE_COVERAGE' AS reject_code,
    'HIGH' AS severity,
    strftime(current_timestamp, '%Y-%m-%dT%H:%M:%SZ') AS detected_ts,
    'R010' AS rule_id
  FROM clean_header h
  JOIN member_norm m ON h.member_id = m.member_id
  WHERE h.service_from_dt IS NOT NULL
    AND h.service_to_dt IS NOT NULL
    AND m.coverage_start_dt IS NOT NULL
    AND m.coverage_end_dt IS NOT NULL
    AND (
      h.service_from_dt < m.coverage_start_dt
      OR h.service_from_dt > m.coverage_end_dt
      OR h.service_to_dt < m.coverage_start_dt
      OR h.service_to_dt > m.coverage_end_dt
    )
),
r011 AS (
  SELECT
    h.claim_id,
    CAST(NULL AS VARCHAR) AS line_id,
    h.batch_id,
    h.lob,
    h.vendor,
    h.service_from,
    'DUPLICATE' AS reject_category,
    'DUP_CLAIM_KEY' AS reject_code,
    'HIGH' AS severity,
    strftime(current_timestamp, '%Y-%m-%dT%H:%M:%SZ') AS detected_ts,
    'R011' AS rule_id
  FROM (
    SELECT
      h.*,
      COUNT(*) OVER (
        PARTITION BY h.batch_id, h.member_id, h.provider_npi, h.service_from, h.total_charge_2dp
      ) AS dup_cnt
    FROM clean_header h
  ) h
  WHERE h.dup_cnt > 1
),
r012 AS (
  SELECT
    h.claim_id,
    CAST(NULL AS VARCHAR) AS line_id,
    h.batch_id,
    h.lob,
    h.vendor,
    h.service_from,
    'FINANCIAL' AS reject_category,
    'PAID_GT_ALLOWED' AS reject_code,
    'MONITOR' AS severity,
    strftime(current_timestamp, '%Y-%m-%dT%H:%M:%SZ') AS detected_ts,
    'R012' AS rule_id
  FROM clean_header h
  WHERE h.total_paid_2dp IS NOT NULL
    AND h.total_allowed_2dp IS NOT NULL
    AND h.total_paid_2dp > h.total_allowed_2dp
),
r013 AS (
  SELECT
    h.claim_id,
    CAST(NULL AS VARCHAR) AS line_id,
    h.batch_id,
    h.lob,
    h.vendor,
    h.service_from,
    'FINANCIAL' AS reject_category,
    'ALLOWED_GT_CHARGE' AS reject_code,
    'MONITOR' AS severity,
    strftime(current_timestamp, '%Y-%m-%dT%H:%M:%SZ') AS detected_ts,
    'R013' AS rule_id
  FROM clean_header h
  WHERE h.total_allowed_2dp IS NOT NULL
    AND h.total_charge_2dp IS NOT NULL
    AND h.total_allowed_2dp > h.total_charge_2dp
)
SELECT * FROM r010
UNION ALL SELECT * FROM r011
UNION ALL SELECT * FROM r012
UNION ALL SELECT * FROM r013;

CREATE OR REPLACE TEMP VIEW rejects_line AS
WITH base AS (
  SELECT
    l.claim_id,
    l.line_id,
    h.batch_id,
    h.lob,
    h.vendor,
    h.service_from,
    l.proc_code,
    l.dx_code,
    l.charge_2dp,
    l.allowed_2dp,
    l.paid_2dp
  FROM line_norm l
  JOIN clean_header h
    ON h.claim_id = l.claim_id
),
r012 AS (
  SELECT
    claim_id, line_id, batch_id, lob, vendor, service_from,
    'FINANCIAL' AS reject_category,
    'PAID_GT_ALLOWED' AS reject_code,
    'MONITOR' AS severity,
    strftime(current_timestamp, '%Y-%m-%dT%H:%M:%SZ') AS detected_ts,
    'R012' AS rule_id
  FROM base
  WHERE paid_2dp IS NOT NULL AND allowed_2dp IS NOT NULL AND paid_2dp > allowed_2dp
),
r013 AS (
  SELECT
    claim_id, line_id, batch_id, lob, vendor, service_from,
    'FINANCIAL' AS reject_category,
    'ALLOWED_GT_CHARGE' AS reject_code,
    'MONITOR' AS severity,
    strftime(current_timestamp, '%Y-%m-%dT%H:%M:%SZ') AS detected_ts,
    'R013' AS rule_id
  FROM base
  WHERE allowed_2dp IS NOT NULL AND charge_2dp IS NOT NULL AND allowed_2dp > charge_2dp
),
r014 AS (
  SELECT
    claim_id, line_id, batch_id, lob, vendor, service_from,
    'CODE_SET' AS reject_category,
    'NULL_PROC' AS reject_code,
    'MONITOR' AS severity,
    strftime(current_timestamp, '%Y-%m-%dT%H:%M:%SZ') AS detected_ts,
    'R014' AS rule_id
  FROM base
  WHERE is_missing(proc_code)
),
r015 AS (
  SELECT
    claim_id, line_id, batch_id, lob, vendor, service_from,
    'CODE_SET' AS reject_category,
    'NULL_DX' AS reject_code,
    'MONITOR' AS severity,
    strftime(current_timestamp, '%Y-%m-%dT%H:%M:%SZ') AS detected_ts,
    'R015' AS rule_id
  FROM base
  WHERE is_missing(dx_code)
)
SELECT * FROM r012
UNION ALL SELECT * FROM r013
UNION ALL SELECT * FROM r014
UNION ALL SELECT * FROM r015;

CREATE OR REPLACE TEMP VIEW rejects_batch_anomaly AS
WITH batch_key_counts AS (
  SELECT
    h.batch_id,
    h.member_id,
    h.provider_npi,
    h.service_from,
    h.total_charge_2dp,
    COUNT(*) AS key_cnt
  FROM header_norm h
  GROUP BY 1,2,3,4,5
),
batch_dup_participation AS (
  SELECT
    h.batch_id,
    max(h.lob) AS lob,
    max(h.vendor) AS vendor,
    COUNT(*) AS claims_in_batch,
    SUM(CASE WHEN k.key_cnt > 1 THEN 1 ELSE 0 END) AS dup_participating_rows
  FROM header_norm h
  JOIN batch_key_counts k
    ON h.batch_id = k.batch_id
   AND coalesce(h.member_id, '') = coalesce(k.member_id, '')
   AND coalesce(h.provider_npi, '') = coalesce(k.provider_npi, '')
   AND coalesce(h.service_from, '') = coalesce(k.service_from, '')
   AND coalesce(CAST(h.total_charge_2dp AS VARCHAR), '') = coalesce(CAST(k.total_charge_2dp AS VARCHAR), '')
  GROUP BY h.batch_id
),
r010_by_batch AS (
  SELECT
    batch_id,
    COUNT(*) AS elig_mismatch_rows
  FROM rejects_header_non_blocker
  WHERE rule_id = 'R010'
  GROUP BY batch_id
),
weekly_volume AS (
  SELECT
    h.batch_id,
    max(h.lob) AS lob,
    max(h.vendor) AS vendor,
    COUNT(*) AS claims_in_batch,
    TRY_CAST(try_strptime(substr(h.batch_id, 7, 8), '%Y%m%d') AS DATE) AS week_start
  FROM header_norm h
  GROUP BY h.batch_id
),
weekly_ranked AS (
  SELECT
    w.*,
    row_number() OVER (PARTITION BY w.lob, w.vendor ORDER BY w.week_start) AS wk_idx
  FROM weekly_volume w
),
weekly_with_median AS (
  SELECT
    w.*,
    (
      SELECT median(prev.claims_in_batch)
      FROM weekly_ranked prev
      WHERE prev.lob = w.lob
        AND prev.vendor = w.vendor
        AND prev.wk_idx BETWEEN w.wk_idx - 8 AND w.wk_idx - 1
    ) AS trailing_8wk_median,
    (
      SELECT COUNT(*)
      FROM weekly_ranked prev
      WHERE prev.lob = w.lob
        AND prev.vendor = w.vendor
        AND prev.wk_idx BETWEEN w.wk_idx - 8 AND w.wk_idx - 1
    ) AS trailing_weeks
  FROM weekly_ranked w
),
r901 AS (
  SELECT
    concat('BATCH::', b.batch_id) AS claim_id,
    CAST(NULL AS VARCHAR) AS line_id,
    b.batch_id,
    b.lob,
    b.vendor,
    CAST(NULL AS VARCHAR) AS service_from,
    'DUPLICATE' AS reject_category,
    'DUP_RATE_GT_1PCT' AS reject_code,
    'HIGH' AS severity,
    strftime(current_timestamp, '%Y-%m-%dT%H:%M:%SZ') AS detected_ts,
    'R901' AS rule_id
  FROM batch_dup_participation b
  WHERE b.claims_in_batch > 0
    AND (CAST(b.dup_participating_rows AS DOUBLE) / CAST(b.claims_in_batch AS DOUBLE)) > 0.01
),
r902 AS (
  SELECT
    concat('BATCH::', b.batch_id) AS claim_id,
    CAST(NULL AS VARCHAR) AS line_id,
    b.batch_id,
    b.lob,
    b.vendor,
    CAST(NULL AS VARCHAR) AS service_from,
    'MEMBER_ELIGIBILITY' AS reject_category,
    'ELIG_MISMATCH_GT_2PCT' AS reject_code,
    'HIGH' AS severity,
    strftime(current_timestamp, '%Y-%m-%dT%H:%M:%SZ') AS detected_ts,
    'R902' AS rule_id
  FROM batch_dup_participation b
  LEFT JOIN r010_by_batch r ON r.batch_id = b.batch_id
  WHERE b.claims_in_batch > 0
    AND (CAST(coalesce(r.elig_mismatch_rows, 0) AS DOUBLE) / CAST(b.claims_in_batch AS DOUBLE)) > 0.02
),
r903 AS (
  SELECT
    concat('BATCH::', w.batch_id) AS claim_id,
    CAST(NULL AS VARCHAR) AS line_id,
    w.batch_id,
    w.lob,
    w.vendor,
    CAST(NULL AS VARCHAR) AS service_from,
    'VOLUME_ANOMALY' AS reject_category,
    'VOLUME_SHIFT_GT_15PCT' AS reject_code,
    'MONITOR' AS severity,
    strftime(current_timestamp, '%Y-%m-%dT%H:%M:%SZ') AS detected_ts,
    'R903' AS rule_id
  FROM weekly_with_median w
  WHERE w.trailing_weeks = 8
    AND w.trailing_8wk_median IS NOT NULL
    AND w.trailing_8wk_median > 0
    AND abs(CAST(w.claims_in_batch AS DOUBLE) - CAST(w.trailing_8wk_median AS DOUBLE)) / CAST(w.trailing_8wk_median AS DOUBLE) > 0.15
)
SELECT * FROM r901
UNION ALL SELECT * FROM r902
UNION ALL SELECT * FROM r903;

CREATE OR REPLACE TEMP VIEW rejects_all AS
SELECT * FROM rejects_header_blocker
UNION ALL
SELECT * FROM rejects_header_non_blocker
UNION ALL
SELECT * FROM rejects_line
UNION ALL
SELECT * FROM rejects_batch_anomaly;

CREATE OR REPLACE TEMP VIEW rejects_final AS
SELECT
  claim_id,
  line_id,
  batch_id,
  lob,
  vendor,
  service_from,
  reject_category,
  reject_code,
  severity,
  detected_ts
FROM rejects_all;

COPY (
  SELECT
    claim_id,line_id,batch_id,lob,vendor,service_from,
    reject_category,reject_code,severity,detected_ts
  FROM rejects_final
  ORDER BY batch_id, claim_id, line_id, reject_code
) TO '{rejects_csv}' (HEADER, DELIMITER ',');
""")

print(f'Wrote: {rejects_csv}')

con.execute(f"""
CREATE OR REPLACE TEMP VIEW triage_enriched AS
SELECT
  CAST(date_trunc('week', TRY_CAST(try_strptime(substr(r.batch_id, 7, 8), '%Y%m%d') AS DATE)) AS DATE) AS week_start,
  r.lob,
  r.vendor,
  r.reject_category,
  r.severity,
  r.reject_code,
  r.claim_id,
  CASE WHEN r.claim_id LIKE 'BATCH::%' THEN 1 ELSE 0 END AS is_batch_anomaly
FROM rejects_final r;

CREATE OR REPLACE TEMP VIEW triage_summary_final AS
SELECT
  week_start,
  lob,
  vendor,
  reject_category,
  severity,
  COUNT(*) AS reject_count,
  SUM(CASE WHEN is_batch_anomaly = 1 THEN 1 ELSE 0 END)
    + COUNT(DISTINCT CASE WHEN is_batch_anomaly = 0 THEN claim_id ELSE NULL END) AS affected_claims
FROM triage_enriched
GROUP BY 1,2,3,4,5;

CREATE OR REPLACE TEMP VIEW triage_by_code_final AS
SELECT
  week_start,
  lob,
  vendor,
  reject_code,
  severity,
  COUNT(*) AS reject_count,
  SUM(CASE WHEN is_batch_anomaly = 1 THEN 1 ELSE 0 END)
    + COUNT(DISTINCT CASE WHEN is_batch_anomaly = 0 THEN claim_id ELSE NULL END) AS affected_claims
FROM triage_enriched
GROUP BY 1,2,3,4,5;

COPY (
  SELECT
    week_start,
    lob,
    vendor,
    reject_category,
    severity,
    reject_count,
    affected_claims
  FROM triage_summary_final
  ORDER BY week_start, lob, vendor, reject_category, severity
) TO '{triage_csv}' (HEADER, DELIMITER ',');
""")

print(f'Wrote: {triage_csv}')

con.execute(f"""
COPY (
  SELECT
    week_start,
    lob,
    vendor,
    reject_code,
    severity,
    reject_count,
    affected_claims
  FROM triage_by_code_final
  ORDER BY week_start, lob, vendor, reject_code, severity
) TO '{triage_by_code_csv}' (HEADER, DELIMITER ',');
""")

print(f'Wrote: {triage_by_code_csv}')

Wrote: C:/Users/halol/Desktop/encounters/outputs/rejects.csv


Wrote: C:/Users/halol/Desktop/encounters/outputs/triage_summary.csv


Wrote: C:/Users/halol/Desktop/encounters/outputs/triage_by_code.csv


In [3]:
severity_rows = con.execute("""
SELECT severity, COUNT(*) AS reject_count
FROM rejects_final
GROUP BY severity
ORDER BY CASE severity WHEN 'BLOCKER' THEN 1 WHEN 'HIGH' THEN 2 WHEN 'MONITOR' THEN 3 ELSE 4 END
""").fetchall()

rule_rows = con.execute("""
SELECT rule_id, COUNT(*) AS reject_count
FROM rejects_all
GROUP BY rule_id
ORDER BY rule_id
""").fetchall()

present_rules = {row[0] for row in rule_rows}
all_rules = [f'R{i:03d}' for i in range(1, 16)]

schema_cols = [r[1] for r in con.execute("PRAGMA table_info('rejects_final')").fetchall()]
expected_cols = [
    'claim_id','line_id','batch_id','lob','vendor','service_from',
    'reject_category','reject_code','severity','detected_ts'
]

# BLOCKER precedence check: claims with any BLOCKER row must not emit R010-R015.
blocker_precedence_violations = con.execute("""
SELECT COUNT(*)
FROM rejects_all r
JOIN (
  SELECT DISTINCT claim_id
  FROM rejects_header_blocker
  WHERE claim_id IS NOT NULL AND trim(claim_id) <> ''
) b ON r.claim_id = b.claim_id
WHERE r.rule_id IN ('R010','R011','R012','R013','R014','R015')
""").fetchone()[0]

# Batch anomaly row convention checks.
anomaly_convention_violations = con.execute("""
SELECT COUNT(*)
FROM rejects_all
WHERE rule_id IN ('R901','R902','R903')
  AND (
    claim_id NOT LIKE 'BATCH::%'
    OR line_id IS NOT NULL
    OR NOT is_missing(service_from)
    OR (rule_id = 'R901' AND reject_category <> 'DUPLICATE')
    OR (rule_id = 'R902' AND reject_category <> 'MEMBER_ELIGIBILITY')
    OR (rule_id = 'R903' AND reject_category <> 'VOLUME_ANOMALY')
  )
""").fetchone()[0]

anomaly_counts = con.execute("""
SELECT rule_id, reject_code, COUNT(*) AS cnt
FROM rejects_all
WHERE rule_id IN ('R901','R902','R903')
GROUP BY 1,2
ORDER BY 1
""").fetchall()

# Manifest-driven coverage expectation for R001-R015.
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
manifest_rules = set()
for incident in manifest.get('incidents', {}).values():
    for key in incident.get('rules', {}).keys():
        m = re.match(r'^(R\d{3})_', key)
        if m:
            manifest_rules.add(m.group(1))
manifest_rules = {r for r in manifest_rules if r in set(all_rules)}

missing_vs_manifest = sorted(r for r in manifest_rules if r not in present_rules)
missing_vs_all = sorted(r for r in all_rules if r not in present_rules)

print('Severity counts:')
for severity, count in severity_rows:
    print(f'  {severity}: {count}')

print()
print('Rule counts (R001-R015):')
rule_count_map = {k: v for k, v in rule_rows}
for rule in all_rules:
    print(f"  {rule}: {rule_count_map.get(rule, 0)}")

print()
print('Rule coverage (R001-R015):')
for rule in all_rules:
    status = 'PRESENT' if rule in present_rules else 'MISSING'
    print(f'  {rule}: {status}')

print()
print('Batch anomaly counts (R901-R903):')
for rule_id, reject_code, cnt in anomaly_counts:
    print(f'  {rule_id} ({reject_code}): {cnt}')

print()
print('Manifest expected rules (R001-R015 subset):', sorted(manifest_rules))
print('Missing vs manifest:', missing_vs_manifest)
print('Missing vs full R001-R015:', missing_vs_all)

print()
print('rejects.csv schema exact match:', schema_cols == expected_cols)
print('Columns:', schema_cols)
print('BLOCKER precedence violations (expected 0):', blocker_precedence_violations)
print('Batch anomaly row convention violations (expected 0):', anomaly_convention_violations)

assert schema_cols == expected_cols, 'rejects_final schema mismatch'
assert blocker_precedence_violations == 0, 'BLOCKER precedence violated: non-BLOCKER rejects emitted for BLOCKER claims'
assert anomaly_convention_violations == 0, 'batch anomaly row conventions violated'
assert len(missing_vs_manifest) == 0, f'missing manifest-driven rules in rejects: {missing_vs_manifest}'
assert len(missing_vs_all) == 0, f'missing required rules R001-R015 in rejects: {missing_vs_all}'

Severity counts:
  BLOCKER: 80
  HIGH: 18
  MONITOR: 121

Rule counts (R001-R015):
  R001: 10
  R002: 5
  R003: 10
  R004: 5
  R005: 10
  R006: 10
  R007: 10
  R008: 10
  R009: 10
  R010: 6
  R011: 10
  R012: 40
  R013: 40
  R014: 20
  R015: 20

Rule coverage (R001-R015):
  R001: PRESENT
  R002: PRESENT
  R003: PRESENT
  R004: PRESENT
  R005: PRESENT
  R006: PRESENT
  R007: PRESENT
  R008: PRESENT
  R009: PRESENT
  R010: PRESENT
  R011: PRESENT
  R012: PRESENT
  R013: PRESENT
  R014: PRESENT
  R015: PRESENT

Batch anomaly counts (R901-R903):
  R901 (DUP_RATE_GT_1PCT): 1
  R902 (ELIG_MISMATCH_GT_2PCT): 1
  R903 (VOLUME_SHIFT_GT_15PCT): 1

Manifest expected rules (R001-R015 subset): ['R001', 'R002', 'R003', 'R004', 'R005', 'R006', 'R007', 'R008', 'R009', 'R010', 'R011', 'R012', 'R013', 'R014', 'R015']
Missing vs manifest: []
Missing vs full R001-R015: []

rejects.csv schema exact match: True
Columns: ['claim_id', 'line_id', 'batch_id', 'lob', 'vendor', 'service_from', 'reject_category', 